In [1]:
import torch
import numpy as np

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import torchvision.transforms as tvf

from unicorrn.model import build_model
from unicorrn.utils import safe_load_weights
from unicorrn.utils.config import read_yaml_config
from fvcore.nn import parameter_count

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
xFormers is available.
Flash attention is not available
Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/projects/vig/prajnan/envs/unicorrn/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
MODEL_CONFIG_PATH="../configs/models/unicorrn_large_stage2.yml"
CKPT_PATH="../pretrained_models/UniCorrn_Large_Stage2.pth"

model_cfg = read_yaml_config(MODEL_CONFIG_PATH)

model = build_model(model_cfg.NAME, cfg=model_cfg)
print(f"Model params: {parameter_count(model)[''] / 1e6:.1f}M")

weights = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
safe_load_weights(model, weights["model"])

Model params: 599.8M
weights safely loaded


In [3]:
def from_numpy(x):
    if isinstance(x, np.ndarray):
        # 0-d object arrays = nested dicts/lists/strings/scalars saved via pickle
        if x.dtype == object:
            return from_numpy(x.item())
        # 0-d numeric arrays = scalars; unwrap to Python number
        if x.ndim == 0:
            return x.item()
        # numeric ndarray -> tensor (skip dtypes torch can't handle)
        if x.dtype.kind in ("b", "i", "u", "f", "c"):
            return torch.from_numpy(x.copy())
        return x  # e.g. string arrays
    if isinstance(x, dict):
        return {k: from_numpy(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(from_numpy(v) for v in x)
    return x


with np.load("../assets/3d3d_sample.npz", allow_pickle=True) as npz:
    sample = {k: from_numpy(npz[k]) for k in npz.files}

sample.keys()

dict_keys(['dataset', 'src_norm_meta', 'min_src_grid_coord', 'target_indices', 'min_tgt_grid_coord', 'query_indices', 'src_grid_coord', 'tgt_pcd', 'tgt_norm_meta', 'src_pcd', 'tgt_grid_coord', 'src_length', 'tgt_length', 'batch_size', 'tgt2src_transform', 'queries', 'norm_queries', 'targets', 'norm_targets'])

In [4]:
sample_ = {}
cuda_keys = [
    "src_pcd",
    "tgt_pcd",
    "src_grid_coord",
    "tgt_grid_coord",
    "src_length",
    "tgt_length",
]
for k, v in sample.items():
    if k in cuda_keys:
        sample_[k] = v.to('cuda')

queries = sample["queries"].cuda()

model.eval()
model = model.cuda()

In [5]:
with torch.no_grad():
    preds = model(task="pcd2pcd", sample=sample_, query_pos=queries)

In [ ]:
queries = sample["queries"].squeeze().cpu()
src_pcd = sample["src_pcd"].cpu()
tgt_pcd = sample["tgt_pcd"].cpu()

targets = sample["norm_targets"].squeeze().cpu()
targets_preds = preds["corr_predictions"].squeeze(0).cpu()

src_points = go.Scatter3d(
    x=src_pcd[...,0],
    y=src_pcd[...,1],
    z=src_pcd[...,2],
    mode='markers',
    name='Source PCD',
    marker=dict(
        size=1.5,
        color=src_pcd[...,2],        # color by z value
        colorscale='Blues_r',
        opacity=1.0
    )
)

stride = 1
src_kpts3d = go.Scatter3d(
    x=queries[::stride][...,0],
    y=queries[::stride][...,1],
    z=queries[::stride][...,2],
    mode='markers',
    name='Queries',
    marker=dict(
        size=7.5,
        color=queries[::stride][...,0],        # color by z value
        colorscale='jet',
        opacity=1.0,
        symbol='x'
    )
)

tgt_points = go.Scatter3d(
    x=tgt_pcd[...,0],
    y=tgt_pcd[...,1],
    z=tgt_pcd[...,2],
    mode='markers',
    name='Target PCD',
    marker=dict(
        size=1.5,
        color=tgt_pcd[...,2],        # color by z value
        colorscale='Blues_r',
        opacity=1.0
    )
)

tgt_kpts3d = go.Scatter3d(
    x=targets[::stride][...,0],
    y=targets[::stride][...,1],
    z=targets[::stride][...,2],
    mode='markers',
    name='Targets GT',
    marker=dict(
        size=5,
        color=targets[::stride][...,0],        # color by x value
        colorscale='jet',
        opacity=0.8,
        symbol='x'
    )
)

tgt_kpts3d_pred = go.Scatter3d(
    x=targets_preds[::stride][...,0],
    y=targets_preds[::stride][...,1],
    z=targets_preds[::stride][...,2],
    mode='markers',
    name='Targets Pred',
    marker=dict(
        size=7.5,
        color=targets[::stride][...,0],        # color by x value
        colorscale='jet',
        opacity=1.0,
        # symbol='x'
    )
)

fig = go.Figure(data=[src_points, tgt_points, src_kpts3d, tgt_kpts3d, tgt_kpts3d_pred])

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='black',
    ),
    paper_bgcolor='black',
    legend=dict(
        x=0,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.6)',
        font=dict(color='white'),
    ),
    margin=dict(l=0, r=0, b=0, t=0),
)

fig.show()